# Data Merging

## Mounting Drive and Importing Libraries

In [1]:
# mounting drive
from google.colab import drive
drive.mount("/content/drive")

# importing libraries
import pandas as pd

Mounted at /content/drive


## Weather Data

In [2]:
# loading weather data
path = "/content/drive/MyDrive/Colab Notebooks/SCC 450 - Group Project/Week 10/Data/Pivoted Weather Data.csv"
df_weather = pd.read_csv(path, parse_dates = ["SDATE"])


## Moth Data

In [3]:
# loading moth data
path = "/content/drive/MyDrive/Colab Notebooks/SCC 450 - Group Project/Week 10/Data/Moth Data Merged.csv"
df_moth = pd.read_csv(path, parse_dates = ["SDATE", "FROMDATE"])

# creating new column "DATE" which captures dates during sampling and adds new rows for rows with SPERIOD_D > 1
df_moth["DATE"] = df_moth.apply(lambda row: row["FROMDATE"] if row["SPERIOD_D"] == 1 else pd.date_range(start = row["FROMDATE"], periods = row["SPERIOD_D"], freq = "D"), axis = 1)
df_moth = df_moth.explode("DATE").reset_index(drop = True)

/tmp/ipython-input-1075859364.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_moth = pd.read_csv(path, parse_dates = ["SDATE", "FROMDATE"])


## Merging the Weather Data and the Moth Data

In [4]:
# merging

# changing SDATE name (due to merge)
df_moth = df_moth.rename(columns = {"SDATE": "SAMPLE_DATE"})

# performing the merge
df_final = df_moth.merge(df_weather, how = "inner", left_on = ["SITECODE", "DATE"], right_on = ["SITECODE", "SDATE"])


# DESCRIPTION:
# moth data is clean
# weather data needs imputing - has missing values !!!

# we wanted to merge moth data with weather data so we can see how weather affects moth sampling
# some samplings were recorded over more than 1 day
    # as such, we created a new column "DATE" which captures dates during sampling - used to match with weather data
    # we created additional rows for samples that used traps which ran for more than 1 day so we can match these entries with weather variables on each day of sampling

# our logic is that FROMDATE (when the sampling starts) is the first day of sampling and SDATE is the collection date
  # as such, we care about weather from FROMDATE up until SDATE (SDATE excluded)

# example: SPERIOD_D = 2, FROMDATE = 11/11/1999, SDATE = 11/13/1999
# the data range we care about in terms of weather (how does if affect the sample) runs on 11/11/1999 and 11/12/1999
# SDATE date is excluded as it is a collection date
# such entry has 2 rows with DATE values 11/11/1999 and 11/12/1999 which are mactched with SDATE (sample date) entries from weather data

## Imputing Missing Values

In [5]:
# dropping year 1992 - there are weather variables that are missing completely (100% for STMP10, STMP30, SURWET)
df_final = df_final[df_final["YEAR"] != 1992].reset_index(drop = True)

# imputing values based on SITECODE and WEEK at each SITECODE using mean
def impute_values_by_week(df, column):
  imputations = df.groupby(["SITECODE", "WEEK_NEW"])[column].transform("mean")
  return imputations

weather_vars = ["ALBGRD",	"DRYTMP", "NETRAD",	"RAIN",	"SOLAR",	"STMP10",	"STMP30",	"SURWET",	"WDIR",	"WSPEED"]

for i in range(len(weather_vars)):
  df_final[weather_vars[i]] = df_final[weather_vars[i]].fillna(impute_values_by_week(df_final, weather_vars[i]))

# 4 rows have missing values after the imputation - dropping them
df_final = df_final.dropna()

# dropping entries where sampling took more than 7 days
df_final = df_final[df_final["SPERIOD_D"] <=  7].reset_index(drop = True)

# removing ALBGRD column - supposed to be ratio
# here, it is captured as W m-2 with lots of outliers - removing entries containing there outliers could harm the dataset a lot - removing the entire column
df_final = df_final.drop(columns = "ALBGRD")

# removing rows with the smallest 1% and largest 1% of values in the distribution of NETRAD
df_final = df_final[(df_final["NETRAD"] <= df_final["NETRAD"].quantile(0.99)) & ((df_final["NETRAD"] >= df_final["NETRAD"].quantile(0.01)))].reset_index(drop = True)

# result
df_final

# outputting the dataset
df_final.to_csv("DATA_CLEAN.csv", index = False)

# outliers in other variables are kept - most of these measure almost identical thing (temperature) and we want to account for realistic "unusual" days


# adjusting the duplicates - collapsing entries with SPERIOD_D > 1 into one row - averaging out weather during that period and summing RAIN values
moth_vars = ["SITECODE", "LCODE", "SAMPLE_DATE", "FIELDNAME", "FROMDATE", "YEAR", "WEEK_NEW", "98_PERCENTILE"]
df_final = df_final.groupby(moth_vars).agg(
    {
    "VALUE": "first",
    "SPERIOD_D": "first",
    "DRYTMP": "mean",
    "NETRAD": "mean",
    "SOLAR": "mean",
    "STMP10": "mean",
    "STMP30": "mean",
    "SURWET": "mean",
    "WDIR": "mean",
    "WSPEED": "mean",
    "RAIN": "sum"}).reset_index()

# outputting the data
df_final.to_csv("DATA_CLEAN(no_moth_duplicates).csv", index = False)

# this dataset does not have any duplicates - if moth sample has SPERIOD_D > 1, its weather data are averaged and recorded as one row